# Test 2 — Public-Benchmark Coref RAG Eval (DAPR ConditionalQA, local)

**Research question:** On standard informative-text retrieval, does *coref-before-embed*
and *hybrid dense+lexical* beat a conventional dense baseline?

Unlike Test 1 (one Wikipedia doc, ~20 self-generated pronoun questions), this test uses a
**recognized public benchmark** with its **fixed passages, queries, and official qrels**.
No custom q-gen, no re-chunking, no LLM-as-judge, no paid embedding API — everything runs
**locally on a free Kaggle T4 GPU**.

## POC setup (single optimal model + dataset)

| Piece | Choice | Why |
|-------|--------|-----|
| **Embedding model** | `BAAI/bge-small-en-v1.5` (dense, 384-d) | English-strong, runs on a free T4, no multi-GB download |
| **Lexical model** | `rank_bm25` (BM25 Okapi) | Classic, recognizable sparse signal for the hybrid |
| **Dataset** | **DAPR ConditionalQA** (`UKPLab/dapr`) | The benchmark *built to reward coreference / document context* |

DAPR ConditionalQA is informative text with strong document context (271 test queries). It follows
the DAPR "the venue" -> "the Half Moon, Putney" pattern: the gold passage refers to an entity by a
reference noun, so the retriever must use document context (coreference).

## Three ingestion variants

| Label | Retrieval |
|-------|-----------|
| **baseline** | Dense on the **original** passage text |
| **coref_dense** | Dense on the **LingMess coref rewrite** (same `_id`) |
| **coref_hybrid** | **RRF** fuse: **BM25(original)** + **dense(coref)**, `RRF_K=60` |

**Invariants**
- The text **returned to the user** is always the **original** (never the coref rewrite).
- Gold = **official qrels only** (`corpus_id` match, `score > 0`).
- Same passage IDs across all variants; 1:1 coref alignment is asserted.

**Metrics:** Recall@5, nDCG@10, MRR (via `pytrec_eval`), plus flip counts (recovered / hurt / both-fail)
vs the dense baseline. Relative comparison across variants matters more than absolute SOTA numbers.
The corpus is capped (whole gold-containing documents first, then top-up) to fit a free T4 —
documented in findings.


## 1. Setup & Install

`sentence-transformers` provides bge-small; `rank_bm25` the lexical signal; `pytrec_eval` the
official TREC metrics. `fastcoref` + pinned `transformers<5` is reused from Test 1 (LingMess needs
the transformers 4.x API).

In [ ]:
!pip install -q datasets pytrec_eval tqdm pandas numpy rank_bm25 sentence-transformers
# fastcoref's LingMessModel targets transformers 4.x. Kaggle ships 5.x, which breaks LingMess. Pin.
!pip install -q "transformers<5" fastcoref
!python -m spacy download en_core_web_sm
# IMPORTANT: after this cell finishes, RESTART THE KERNEL once so pinned transformers 4.x is the
# version actually imported, then run the cells below (skip re-running this one).
# NOTE: pip may print dependency-conflict WARNINGS about numba / cudf / cuml (Kaggle's RAPIDS
# packages, unused here). Safe to ignore.

## 2. Config

No API keys required — bge-small runs locally on the Kaggle T4. Dense vectors are cached to disk per
`(dataset, variant, role, content-hash)` so the notebook is **resume-friendly**.

In [ ]:
import os, logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("coref_public_eval")

# --- Model: single optimal choice for a free-T4 POC ---
# bge-small-en-v1.5 is English-strong dense retrieval, 384-d, no multi-GB download.
# The lexical half of the hybrid is BM25 (rank_bm25).
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
BATCH_SIZE  = 64                # small model -> large batches are fine on a T4
MAX_LEN     = 512               # bge-small context window

# --- Core config ---
TOP_K       = 5                 # primary hit cutoff (Recall@5, flips)
RRF_K       = 60                # reciprocal-rank-fusion constant
CACHE_DIR   = "./embed_cache"
RUN_DEPTH   = 100               # keep top-100 per query for metric computation

# --- Coref windowing (LingMess Longformer length limit) ---
COREF_WINDOW_WORDS = 350

# --- Corpus cap (documented in findings): keeps free-T4 runtime + memory sane. ---
# Full ConditionalQA = ~69k passages/coref calls (too heavy on a free T4). We keep whole
# gold-containing documents first (real same-doc distractors), then top up to this many passages.
CONDITIONALQA_MAX_CORPUS = 8000

os.makedirs(CACHE_DIR, exist_ok=True)
log.info("Config | model=%s | TOP_K=%d | RRF_K=%d | batch=%d | cap=%d | cache=%s",
         EMBED_MODEL, TOP_K, RRF_K, BATCH_SIZE, CONDITIONALQA_MAX_CORPUS, CACHE_DIR)

## 3. Coreference engine (LingMess, local) — reused from Test 1

Same LingMess setup as Test 1: force **eager** attention (Longformer is incompatible with SDPA on
transformers 5.x), resolve each pronoun mention to the first **named** mention in its cluster, and
apply char-offset edits right-to-left.

In [ ]:
import re, time

SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
PRONOUN_RE = re.compile(
    r"\b(he|she|him|her|his|hers|they|them|their|theirs|it|its)\b", re.IGNORECASE,
)

def split_sentences(text):
    return [s.strip() for s in SENT_SPLIT_RE.split(text.strip()) if s.strip()]

# LingMess (Longformer) is incompatible with SDPA; transformers 5.x raises instead of falling back.
# Force eager on the config the model init reads. Guard flag on the class prevents double-wrapping.
from transformers.models.auto.modeling_auto import AutoModel

if not getattr(AutoModel, "_coref_eager_patched", False):
    _orig_from_config = AutoModel.from_config.__func__

    @classmethod
    def _from_config_eager(cls, config, **kwargs):
        try:
            config._attn_implementation = "eager"
        except Exception:
            pass
        kwargs.setdefault("attn_implementation", "eager")
        try:
            return _orig_from_config(cls, config, **kwargs)
        except (TypeError, ValueError):
            kwargs.pop("attn_implementation", None)
            return _orig_from_config(cls, config, **kwargs)

    AutoModel.from_config = _from_config_eager
    AutoModel._coref_eager_patched = True

from fastcoref import LingMessCoref

_has_cuda = False
try:
    import torch
    _has_cuda = torch.cuda.is_available()
except Exception:
    pass

coref_model = LingMessCoref(device="cuda" if _has_cuda else "cpu")
log.info("LingMessCoref loaded | device=%s", "cuda" if _has_cuda else "cpu")

PRONOUN_SET = {
    "he", "she", "him", "her", "his", "hers", "they", "them", "their", "theirs",
    "it", "its", "himself", "herself", "themselves", "itself",
}

def _is_pronoun_span(text):
    return text.strip().lower() in PRONOUN_SET

def _has_pronoun_token(text):
    return any(w.strip(".,;:'\"").lower() in PRONOUN_SET for w in text.split())

def _pick_representative(spans_text):
    """Best cluster 'name': prefer proper-noun, pronoun-free spans; else any pronoun-free span;
    else None (never reinject a pronoun). Shortest candidate avoids trailing clauses."""
    proper = [s for s in spans_text
              if not _has_pronoun_token(s) and any(t[:1].isupper() for t in s.split())]
    clean = [s for s in spans_text if not _has_pronoun_token(s)]
    pool = proper or clean
    if not pool:
        return None
    return min(pool, key=len).strip()

def coref_edits(text):
    """Sorted (start, end, replacement) char-offset edits replacing pronouns with cluster names."""
    if not text.strip():
        return []
    preds = coref_model.predict(texts=[text])
    if not preds:                       # LingMess returns [] if text exceeds its length limit
        return []
    clusters = preds[0].get_clusters(as_strings=False)
    edits = []
    for cluster in clusters:
        spans_text = [text[s:e] for s, e in cluster]
        rep = _pick_representative(spans_text)
        if not rep:
            continue
        for (s, e), st in zip(cluster, spans_text):
            if _is_pronoun_span(st) and st.strip().lower() != rep.lower():
                repl = rep + "'s" if st.strip().lower() in {"his", "her", "its", "their", "hers", "theirs"} else rep
                edits.append((s, e, repl))
    return sorted(edits, key=lambda x: x[0])

def apply_edits(text, edits):
    out = text
    for s, e, repl in sorted(edits, key=lambda x: x[0], reverse=True):
        out = out[:s] + repl + out[e:]
    return out

def resolve_coref_text(text):
    """Passage-level coref: replace pronoun mentions with their representative name."""
    return apply_edits(text, coref_edits(text))

### Document-level coref (DAPR passages carry `doc_id`)

DAPR passages carry a `doc_id` and `paragraph_no`, so we resolve coref with **document context**
(a pronoun's antecedent may live in an earlier passage) and map the rewrite back to each passage.
We coref the concatenated passages of a document in word-budgeted windows and route char-offset
edits back to the owning passage — preserving the **official passage boundaries** exactly (1:1, no
re-chunking).

In [ ]:
def coref_passages_doc_level(passages, window_words=COREF_WINDOW_WORDS):
    """passages: list of dicts with 'text' in document order (same doc_id).
    Returns list of coref-rewritten texts, 1:1 aligned with input passages."""
    texts = [p["text"] for p in passages]
    out = list(texts)

    windows, cur, cur_w = [], [], 0
    for i, t in enumerate(texts):
        w = len(t.split())
        if cur and cur_w + w > window_words:
            windows.append(cur); cur, cur_w = [], 0
        cur.append(i); cur_w += w
    if cur:
        windows.append(cur)

    for win in windows:
        spans, pos, parts = [], 0, []
        for k, idx in enumerate(win):
            if k > 0:
                parts.append(" "); pos += 1
            start = pos
            parts.append(texts[idx]); pos += len(texts[idx])
            spans.append((idx, start, pos))
        joined = "".join(parts)
        edits = coref_edits(joined)
        by_p = {}
        for (s, e, repl) in edits:
            for (idx, ss, se) in spans:
                if ss <= s and e <= se:
                    by_p.setdefault(idx, []).append((s - ss, e - ss, repl))
                    break
        for idx, local in by_p.items():
            out[idx] = apply_edits(texts[idx], local)
    return out

## 4. Encoder (bge-small dense, local) with disk cache

We encode passages/queries with `bge-small-en-v1.5` (dense, L2-normalized) and cache vectors to
`CACHE_DIR` keyed by `(dataset, variant, role, content-hash)` so re-runs and resumes skip
re-encoding. The lexical half of the hybrid is BM25, built on demand in the retrieval step (cheap,
CPU-side, nothing to cache).

In [ ]:
import hashlib, pickle
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

def _hash_texts(texts):
    h = hashlib.sha1()
    for t in texts:
        h.update(t.encode("utf-8", "ignore")); h.update(b"\x00")
    return h.hexdigest()[:16]

def _cache_file(dataset, variant, role, texts):
    key = f"{dataset}__{variant}__{role}__{_hash_texts(texts)}"
    return os.path.join(CACHE_DIR, key + ".pkl")

# ---- Load the dense encoder once. ----
dense_model = SentenceTransformer(EMBED_MODEL, device="cuda" if _has_cuda else "cpu")
log.info("Loaded %s (dense) + rank_bm25 (lexical) | device=%s",
         EMBED_MODEL, "cuda" if _has_cuda else "cpu")

def encode_cached(dataset, variant, role, texts):
    """Encode dense vectors with disk cache. role in {'passage','query'}. Returns np.ndarray."""
    cf = _cache_file(dataset, variant, role, texts)
    if os.path.exists(cf):
        with open(cf, "rb") as f:
            arr = pickle.load(f)
        log.info("cache HIT  | %s", os.path.basename(cf))
        return arr
    log.info("cache MISS | encoding %d %s texts (%s/%s)", len(texts), role, dataset, variant)
    arr = dense_model.encode(texts, batch_size=BATCH_SIZE,
                             normalize_embeddings=True, show_progress_bar=False).astype("float32")
    with open(cf, "wb") as f:
        pickle.dump(arr, f)
    return arr

## 5. Retrieval: dense, BM25 lexical, and RRF hybrid

- **Dense**: cosine (dot on normalized vectors) — `baseline` and `coref_dense`.
- **Lexical**: BM25 (`rank_bm25`) over the **original** text — the lexical half of `coref_hybrid`.
- **Hybrid**: **RRF** fuse of `BM25(original)` ranks and `dense(coref)` ranks, `RRF_K=60`.

In [ ]:
def dense_ranklist(q_dense, p_dense, depth=RUN_DEPTH):
    """Return, per query, a list of (passage_idx, score) sorted desc, top-`depth`."""
    sims = q_dense @ p_dense.T                       # (nq, np)
    order = np.argsort(-sims, axis=1)[:, :depth]
    return [[(int(j), float(sims[qi, j])) for j in order[qi]] for qi in range(sims.shape[0])]

def bm25_ranklist(query_texts, passage_texts, depth=RUN_DEPTH):
    """Classic BM25 Okapi lexical ranker (the sparse half of the hybrid)."""
    from rank_bm25 import BM25Okapi
    tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    bm25 = BM25Okapi([tok(t) for t in passage_texts])
    out = []
    for q in query_texts:
        scores = bm25.get_scores(tok(q))
        order = np.argsort(-scores)[:depth]
        out.append([(int(j), float(scores[j])) for j in order])
    return out

def rrf_fuse(ranklists, k=RRF_K, depth=RUN_DEPTH):
    """Reciprocal-rank fusion of multiple per-query rank lists. ranklists: list over systems,
    each = list over queries of [(pidx, score), ...]. Returns fused [(pidx, rrf_score), ...]."""
    nq = len(ranklists[0])
    fused = []
    for qi in range(nq):
        acc = {}
        for sys_rl in ranklists:
            for rank, (pidx, _s) in enumerate(sys_rl[qi]):
                acc[pidx] = acc.get(pidx, 0.0) + 1.0 / (k + rank + 1)
        ordered = sorted(acc.items(), key=lambda x: -x[1])[:depth]
        fused.append([(int(p), float(s)) for p, s in ordered])
    return fused

## 6. Metrics (official qrels, `pytrec_eval`)

We build a TREC-style `run` (`query_id -> {passage_id: score}`) per variant and score with
`pytrec_eval` against the **official qrels** (unchanged). Reported: **Recall@5**, **nDCG@10**,
**MRR**. A per-query **hit** = any retrieved id (top-`TOP_K`) is in that query's qrels with
`score > 0`. Flip analysis compares each coref variant to the dense baseline.

In [ ]:
import pytrec_eval

def build_run(ranklist, query_ids, passage_ids):
    """ranklist: per-query [(pidx, score)]. Returns run dict for pytrec_eval."""
    run = {}
    for qi, qid in enumerate(query_ids):
        run[qid] = {passage_ids[pidx]: float(score) for pidx, score in ranklist[qi]}
    return run

def eval_run(run, qrels, k=TOP_K):
    """Return dict of aggregate metrics + per-query hit set (top-k)."""
    measures = {"recall_5", "ndcg_cut_10", "recip_rank"}
    ev = pytrec_eval.RelevanceEvaluator(qrels, measures)
    per = ev.evaluate(run)
    recall = float(np.mean([v["recall_5"] for v in per.values()])) if per else 0.0
    ndcg = float(np.mean([v["ndcg_cut_10"] for v in per.values()])) if per else 0.0
    mrr = float(np.mean([v["recip_rank"] for v in per.values()])) if per else 0.0

    gold = {q: {d for d, s in rels.items() if s > 0} for q, rels in qrels.items()}
    hits = set()
    for qid, docs in run.items():
        topk = [d for d, _ in sorted(docs.items(), key=lambda x: -x[1])[:k]]
        if gold.get(qid) and (set(topk) & gold[qid]):
            hits.add(qid)
    return {"recall_at_5": recall, "ndcg_at_10": ndcg, "mrr": mrr, "hits": hits, "n": len(qrels)}

## 7. Dataset loader (fixed passages, queries, official qrels)

The loader returns a uniform structure:
```
{ "name", "passages": [{"_id","text","doc_id","order"}], "queries": {qid: text}, "qrels": {qid:{pid:score}} }
```
No custom chunking — we use DAPR's own passage units and official qrels verbatim. DAPR passages
carry `doc_id` + `paragraph_no`, which enables **document-level** coref. When capped, we keep
**whole documents that contain a gold passage first** (gold + its same-document distractors — the
topically near-identical passages that make DAPR a real retrieval test), then top up with other
whole documents up to `CONDITIONALQA_MAX_CORPUS`.

In [ ]:
from datasets import load_dataset
from collections import OrderedDict

def _restrict_corpus(passages, qrels, cap):
    """Document-aware cap that preserves real retrieval difficulty.

    DAPR is hard because the gold passage sits among *topically near-identical* passages from the
    SAME document (same vocabulary), so the retriever must find the specific paragraph, not just
    the right topic. To keep that difficulty we include **whole documents that contain a gold
    passage first** (gold + its same-document distractors), then top up with other whole documents
    until we reach `cap`. This makes the capped pool the *right* subset, not an arbitrary one.
    """
    gold_ids = {pid for rels in qrels.values() for pid, s in rels.items() if s > 0}

    by_doc = OrderedDict()
    for p in passages:
        by_doc.setdefault(p["doc_id"], []).append(p)

    gold_docs = [d for d, ps in by_doc.items() if any(p["_id"] in gold_ids for p in ps)]
    other_docs = [d for d in by_doc if d not in set(gold_docs)]

    kept, seen = [], set()
    def add_doc(d):
        for p in by_doc[d]:
            if p["_id"] not in seen:
                kept.append(p); seen.add(p["_id"])

    for d in gold_docs:                       # 1) all gold-containing docs in full
        add_doc(d)
    for d in other_docs:                      # 2) top up with other whole docs
        if len(kept) >= cap:
            break
        add_doc(d)
    if len(kept) < len(gold_ids):             # 3) safety net: never drop a gold passage
        for p in passages:
            if p["_id"] in gold_ids and p["_id"] not in seen:
                kept.append(p); seen.add(p["_id"])

    log.info("cap | kept=%d passages across %d docs (%d gold docs) | gold passages=%d",
             len(kept), len({p["doc_id"] for p in kept}), len(gold_docs), len(gold_ids))
    return kept

def load_dapr(dataset_name="ConditionalQA", cap=None):
    """DAPR (UKPLab/dapr): passages carry doc_id + paragraph_no for document-level coref."""
    corpus = load_dataset("UKPLab/dapr", f"{dataset_name}-corpus", split="test")
    queries_ds = load_dataset("UKPLab/dapr", f"{dataset_name}-queries", split="test")
    qrels_ds = load_dataset("UKPLab/dapr", f"{dataset_name}-qrels", split="test")

    qrels = {}
    for r in qrels_ds:
        if int(r["score"]) > 0:
            qrels.setdefault(str(r["query_id"]), {})[str(r["corpus_id"])] = int(r["score"])
    qtext = {str(q["_id"]): (q["text"] or "") for q in queries_ds}
    queries = {qid: qtext[qid] for qid in qrels if qid in qtext}

    passages = [{"_id": str(c["_id"]), "text": (c["text"] or ""),
                 "doc_id": str(c.get("doc_id") or c["_id"]),
                 "order": int(c.get("paragraph_no") or 0)}
                for c in corpus]
    if cap:
        passages = _restrict_corpus(passages, qrels, cap)
    log.info("[DAPR %s] passages=%d docs=%d queries=%d qrels=%d",
             dataset_name, len(passages), len({p["doc_id"] for p in passages}),
             len(queries), len(qrels))
    return {"name": f"dapr_{dataset_name}", "passages": passages,
            "queries": queries, "qrels": qrels}

## 8. Build coref rewrites (1:1 with passage IDs)

We produce a `coref_text` per passage, aligned 1:1 with the original by `_id`, using **document-level**
coref (passages of a doc resolved together, in `paragraph_no` order). Pronoun counts before/after
are logged, and the 1:1 ID alignment is asserted. Results are cached so coref runs once.

In [ ]:
def build_coref_texts(ds):
    """Add 'coref_text' to each passage. Returns (pron_before, pron_after)."""
    cache = os.path.join(CACHE_DIR, f"coref__{ds['name']}.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f:
            mapping = pickle.load(f)
        for p in ds["passages"]:
            p["coref_text"] = mapping.get(p["_id"], p["text"])
        log.info("[%s] coref cache HIT (%d passages)", ds["name"], len(mapping))
    else:
        from collections import defaultdict
        groups = defaultdict(list)
        for p in ds["passages"]:
            groups[p["doc_id"]].append(p)
        t0 = time.time()
        for did, plist in tqdm(groups.items(), desc=f"coref {ds['name']}", mininterval=2.0):
            plist.sort(key=lambda x: x["order"])
            if len(plist) > 1:
                rewrites = coref_passages_doc_level(plist)
                for p, rw in zip(plist, rewrites):
                    p["coref_text"] = rw
            else:
                plist[0]["coref_text"] = resolve_coref_text(plist[0]["text"])
        mapping = {p["_id"]: p["coref_text"] for p in ds["passages"]}
        with open(cache, "wb") as f:
            pickle.dump(mapping, f)
        log.info("[%s] coref built | %.1fs | %d passages", ds["name"], time.time() - t0, len(mapping))

    assert all("coref_text" in p for p in ds["passages"]), "coref alignment broken"
    orig_all = " ".join(p["text"] for p in ds["passages"])
    coref_all = " ".join(p["coref_text"] for p in ds["passages"])
    pb, pa = len(PRONOUN_RE.findall(orig_all)), len(PRONOUN_RE.findall(coref_all))
    log.info("[%s] pronouns %d -> %d (%.0f%% reduction)",
             ds["name"], pb, pa, 100 * (1 - pa / max(pb, 1)))
    return pb, pa

## 9. Run the three variants on a dataset

- **baseline**: dense(original) ranks.
- **coref_dense**: dense(coref) ranks.
- **coref_hybrid**: RRF( BM25(original), dense(coref) ).

Queries encoded once; passages encoded per variant (dense original, dense coref) — all cached.
Returns metrics + runs for flip analysis.

In [ ]:
def run_dataset(ds):
    name = ds["name"]
    passages = ds["passages"]
    p_ids = [p["_id"] for p in passages]
    q_ids = list(ds["queries"].keys())
    q_texts = [ds["queries"][q] for q in q_ids]
    orig_texts = [p["text"] for p in passages]
    coref_texts = [p["coref_text"] for p in passages]

    # ---- Dense encodes (cached) ----
    q_dense       = encode_cached(name, "query", "query", q_texts)
    p_dense_orig  = encode_cached(name, "baseline", "passage", orig_texts)
    p_dense_coref = encode_cached(name, "coref_dense", "passage", coref_texts)

    # ---- Rank lists ----
    rl_baseline = dense_ranklist(q_dense, p_dense_orig)
    rl_corefden = dense_ranklist(q_dense, p_dense_coref)
    rl_bm25     = bm25_ranklist(q_texts, orig_texts)        # lexical over ORIGINAL text
    rl_hybrid   = rrf_fuse([rl_bm25, rl_corefden])          # BM25(orig) + dense(coref)

    variants = {
        "baseline":     rl_baseline,
        "coref_dense":  rl_corefden,
        "coref_hybrid": rl_hybrid,
    }
    results, runs = {}, {}
    for label, rl in variants.items():
        run = build_run(rl, q_ids, p_ids)
        runs[label] = run
        results[label] = eval_run(run, ds["qrels"])
        r = results[label]
        log.info("[%s/%s] R@5=%.3f nDCG@10=%.3f MRR=%.3f",
                 name, label, r["recall_at_5"], r["ndcg_at_10"], r["mrr"])
    return {"name": name, "results": results, "runs": runs,
            "n_passages": len(passages), "n_queries": len(q_ids)}

## 10. Results table + flip analysis

`summarize(run_out)` produces the per-dataset table (baseline / coref_dense / coref_hybrid with
Δ vs baseline). `flips(run_out)` gives the flip counts (recovered / hurt / both-fail) for each
coref variant relative to the dense baseline.

In [ ]:
import pandas as pd

def summarize(run_out):
    res = run_out["results"]
    base = res["baseline"]
    rows = []
    for label in ["baseline", "coref_dense", "coref_hybrid"]:
        r = res[label]
        rows.append({
            "variant": label,
            "recall@5": round(r["recall_at_5"], 4),
            "nDCG@10": round(r["ndcg_at_10"], 4),
            "MRR": round(r["mrr"], 4),
            "Δrecall@5": round(r["recall_at_5"] - base["recall_at_5"], 4),
            "ΔnDCG@10": round(r["ndcg_at_10"] - base["ndcg_at_10"], 4),
        })
    df = pd.DataFrame(rows)
    print(f"\n=== {run_out['name']} | passages={run_out['n_passages']} queries={run_out['n_queries']} ===")
    print(df.to_string(index=False))
    return df

def flips(run_out):
    res = run_out["results"]
    base_hits = res["baseline"]["hits"]
    all_qids = set(run_out["runs"]["baseline"].keys())
    out = {}
    for label in ["coref_dense", "coref_hybrid"]:
        h = res[label]["hits"]
        recovered = h - base_hits            # baseline miss -> variant hit
        hurt = base_hits - h                 # baseline hit -> variant miss
        both_fail = all_qids - base_hits - h
        out[label] = {"recovered": len(recovered), "hurt": len(hurt),
                      "both_fail": len(both_fail),
                      "recovered_ids": list(recovered)[:10], "hurt_ids": list(hurt)[:10]}
        print(f"\n[{run_out['name']}] {label} vs baseline: "
              f"recovered={len(recovered)} hurt={len(hurt)} both_fail={len(both_fail)}")
    return out

## 11. Run — DAPR ConditionalQA (all 3 variants)

The benchmark designed to reward document-context understanding (coreference), so it is the core
test of the research question. Corpus capped to whole gold-containing documents first; cache makes
re-runs fast.

In [ ]:
ALL_RUNS = {}     # name -> run_out, collected for the findings report
COREF_STATS = {}  # name -> (pron_before, pron_after)

ds_cqa = load_dapr("ConditionalQA", cap=CONDITIONALQA_MAX_CORPUS)
COREF_STATS["dapr_ConditionalQA"] = build_coref_texts(ds_cqa)
run_cqa = run_dataset(ds_cqa)
ALL_RUNS["dapr_ConditionalQA"] = run_cqa
df_cqa = summarize(run_cqa)
flips_cqa = flips(run_cqa)

## 12. Worked example — a document-context recovery case

Find a query where `coref_dense` recovered a baseline miss, and show the gold passage's **original**
text (returned to the user) vs its **coref rewrite** (embedded). This is the DAPR "the venue" ->
"the Half Moon, Putney" pattern: the answer passage refers to the entity by a reference noun, and
coref injects the name into the embedded text.

In [ ]:
def show_worked_example(ds, run_out, variant="coref_dense"):
    res = run_out["results"]
    recovered = res[variant]["hits"] - res["baseline"]["hits"]
    if not recovered:
        print(f"No recovery for {variant} on {run_out['name']}."); return
    qid = sorted(recovered)[0]
    gold_ids = {pid for pid, s in ds["qrels"][qid].items() if s > 0}
    by_id = {p["_id"]: p for p in ds["passages"]}
    print("QUERY:", ds["queries"][qid])
    print("\nRecovered by:", variant)
    for gid in list(gold_ids)[:1]:
        p = by_id.get(gid)
        if not p:
            continue
        print("\n--- Gold passage", gid, "---")
        print("RETURNED (original):", p["text"][:500])
        print("\nEMBEDDED (coref)   :", p["coref_text"][:500])
        print("\npronouns  orig:", len(PRONOUN_RE.findall(p["text"])),
              "| coref:", len(PRONOUN_RE.findall(p["coref_text"])))

show_worked_example(ds_cqa, ALL_RUNS["dapr_ConditionalQA"])

## 13. Write `test-2-findings.md`

Assembles setup, the result table, Δ vs baseline, flip counts, and scope notes into the findings
file. Re-run after the dataset completes for a full report.

In [ ]:
def _md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |",
             "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

def write_findings(path="test-2-findings.md"):
    out = []
    out.append("# Test 2 — Public-Benchmark Coref RAG Eval (DAPR ConditionalQA, local)\n")
    out.append(f"**Run date:** {time.strftime('%Y-%m-%d')}  ")
    out.append("**Notebook:** `coref_public_eval.ipynb`  ")
    out.append("**Question:** On standard informative-text retrieval, does coref-before-embed "
               "and hybrid dense+lexical beat a conventional dense baseline?\n")
    out.append("## Setup\n")
    out.append(f"- **Embedding model:** {EMBED_MODEL} (dense, local GPU) + rank_bm25 (lexical)")
    out.append(f"- **Variants:** baseline (dense original) / coref_dense (dense LingMess rewrite) / "
               f"coref_hybrid (RRF: BM25(original) + dense(coref), RRF_K={RRF_K})")
    out.append("- **Coref:** fastcoref LingMess, document-level (DAPR passages carry doc_id)")
    out.append(f"- **Metrics:** Recall@5, nDCG@10, MRR (pytrec_eval); TOP_K={TOP_K}")
    out.append("- **Gold:** official qrels only (`score > 0`); passage IDs identical across variants (1:1 coref alignment asserted)")
    out.append(f"- **Corpus cap:** whole gold-containing documents first, topped up to <= {CONDITIONALQA_MAX_CORPUS} passages to fit a free T4.\n")

    for name, run_out in ALL_RUNS.items():
        df = summarize(run_out)
        out.append(f"## {name}\n")
        out.append(f"Passages: {run_out['n_passages']} | Queries: {run_out['n_queries']}")
        pb, pa = COREF_STATS.get(name, (0, 0))
        if pb:
            out.append(f"Pronouns before->after coref: {pb} -> {pa} "
                       f"({100*(1-pa/max(pb,1)):.0f}% reduction)\n")
        out.append(_md_table(df) + "\n")
        base_hits = run_out["results"]["baseline"]["hits"]
        all_qids = set(run_out["runs"]["baseline"].keys())
        out.append("**Flips vs baseline**\n")
        out.append("| variant | recovered | hurt | both_fail |")
        out.append("| --- | --- | --- | --- |")
        for label in ["coref_dense", "coref_hybrid"]:
            h = run_out["results"][label]["hits"]
            out.append(f"| {label} | {len(h - base_hits)} | {len(base_hits - h)} | "
                       f"{len(all_qids - base_hits - h)} |")
        out.append("")

    out.append("## Scope notes\n")
    out.append("- **DAPR ConditionalQA** = document-context / informative retrieval; the benchmark "
               "built to reward coreference understanding.")
    out.append("- **Hybrid** = BM25(original) + dense(coref) fused via RRF.")
    out.append(f"- Corpus subsampled for a free-T4 POC: whole gold-containing documents first "
               f"(gold + their same-document distractors), topped up to <= {CONDITIONALQA_MAX_CORPUS} "
               f"passages. A subset, not the full ~69k corpus, but the right subset for measuring retrieval.")
    out.append("- Relative comparison across variants matters more than absolute SOTA numbers.")
    out.append("- No LLM-as-judge, no answer generation, no custom q-gen, no re-chunking of the public corpus.")

    text = "\n".join(out) + "\n"
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    log.info("Wrote %s (%d chars)", path, len(text))
    print(text[:1500])

write_findings()